## Dream 2000 Tablets Scrapping

In [1]:
import selenium 

In [ ]:

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import time

## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


## -- Tablets List --
Tablets = []


## -- Scraping --
driver = webdriver.Chrome(options=options)
driver.get("https://dream2000.com/en/collections/tablets")
time.sleep(5)

page = 1

while True:

    page_products = 0

    ## -- Product Cards --
    cards = driver.find_elements(By.CSS_SELECTOR,'.product-card__wrapper.color-scheme-theme.anm-reveal-item.overflow-hidden')

    for card in cards:

        try:
            Product = card.find_element(By.CSS_SELECTOR, 'a[href*="/en/products/"]')

            Name = Product.get_attribute('aria-label')
            Link = Product.get_attribute('href')

            Image = card.find_element(By.CSS_SELECTOR, 'img.product-card__image').get_attribute('src')

        except NoSuchElementException:
            continue

        ## -- Price --
        try:
            Price = card.find_element(By.CSS_SELECTOR, '.price').text

        except NoSuchElementException:
            Price = "Product Not Available"

        ## -- Stock --
        try:
            Stock = card.find_element(By.CSS_SELECTOR,'[data-inventory-status]').get_attribute('data-inventory-status')

        except NoSuchElementException:
            Stock = "Product Not Available"

        Tablets.append({
            'Name': Name,
            'Link': Link,
            'Image': Image,
            'Price': Price,
            'Stock': Stock
        })

        page_products += 1

    ## -- Page Summary --

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Tablets)}"
    )

    ## -- Next Page --
    next_buttons = driver.find_elements(
        By.CSS_SELECTOR, "a[aria-label='Next']"
    )

    if not next_buttons:
        print("No Next page. Scraping finished.")
        break

    next_url = next_buttons[0].get_attribute('href')

    if not next_url or next_url == driver.current_url:
        print("Invalid or unchanged Next URL.")
        break

    print("Navigating to:", next_url)

    driver.get(next_url)

    page += 1
    time.sleep(5)


## -- Final Result --
print(f"\nScraping completed. Total products: {len(Tablets)}")


In [ ]:
Tablets

## Import To CSV & Parquet File

In [4]:
import pandas as pd

In [5]:
df_Tablets = pd.DataFrame(Tablets)

In [6]:
df_Tablets.head()

,Name,Link,Image,Price,Stock
0,"Huawei MatePad 11.5"" 2025 WiFi with Smart Keyb...",https://dream2000.com/en/products/matepad-11-5...,https://dream2000.com/cdn/shop/files/2025-09-0...,"LE 36,990.00\nLE 38,900.00",in-stock
1,Redmi Pad 2,https://dream2000.com/en/products/redmi-pad-2-...,https://dream2000.com/cdn/shop/files/4_b2adeda...,"LE 13,990.00\nLE 14,700.00",low-stock
2,Infinix XPad 30E - 4GB RAM - 128GB,https://dream2000.com/en/products/infinix-xpad...,https://dream2000.com/cdn/shop/files/11_304f77...,"LE 9,990.00\nLE 10,500.00",in-stock
3,Lenovo Tab One LTE,https://dream2000.com/en/products/lenovo-tab-o...,https://dream2000.com/cdn/shop/files/2025-06-3...,"LE 7,990.00\nLE 8,400.00",in-stock
4,"Infinix XPAD 30 Pro- 11 Inch - 256GB, 8GB Ram,4G",https://dream2000.com/en/products/infinix-xpad...,https://dream2000.com/cdn/shop/files/rtgdgdfgd...,"LE 19,990.00\nLE 21,000.00",low-stock


In [7]:
df_Tablets.to_csv("Tablets.csv" , index = False)
df_Tablets.to_parquet("Tablets.parquet" , index = False)